# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a guide for loading and exploring the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a [Croissant schema URL](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json).

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Display dataset metadata (name and description)
ds_metadata = dataset.metadata
print(f"Dataset Title: {ds_metadata.name}\n\nDescription: {ds_metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

The dataset may contain several record sets. We'll list all available record sets and their fields with their `@id` for later referencing.

In [ ]:
# List all record sets and their fields/columns by `@id`
record_sets = dataset.metadata.record_sets
print(f"Number of record sets: {len(record_sets)}\n")

for rs in record_sets:
    print(f"Record Set: {rs.name}")
    print(f"  @id: {rs.id}")
    print(f"  Description: {rs.description if hasattr(rs, 'description') else '-'}")
    cols = getattr(rs, 'fields', None)
    if not cols:
        # Try columns as well for tabular record sets
        cols = getattr(rs, 'columns', None)
    if cols:
        print("  Fields/Columns (@id and name):")
        for f in cols:
            print(f"    - @id: {f.id}, name: {f.name}")
    else:
        print("  (No fields/columns found)")
    print()
# Store the record set IDs for later use
record_set_ids = [rs.id for rs in record_sets]

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis.

We'll load records from the first available main record set, using only `@id` references for record set and fields. Adjust the `record_set_id` if you wish to use a different one.

In [ ]:
# Use the first found record set for demonstration
if len(record_set_ids) == 0:
    raise RuntimeError("No record sets are defined in this dataset.")
main_record_set_id = record_set_ids[0]
print(f"Reading records from record set: {main_record_set_id}\n")

records = list(dataset.records(record_set=main_record_set_id))

# Load into DataFrame
main_df = pd.DataFrame(records)
print(f"Columns in the DataFrame (fields/columns by @id):\n{list(main_df.columns)}\n")
main_df.head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing/grouping data.

We'll choose a numeric field (by `@id`) for demonstration. Ensure you use a numeric field available in your DataFrame (see the columns above).

In [ ]:
# Find a numeric field for demonstration
import numpy as np

# Try to automatically pick a float or int column
numeric_fields = [c for c in main_df.columns if pd.api.types.is_numeric_dtype(main_df[c])]
if not numeric_fields:
    # Try columns that look like a numeric type (sometimes loaded as string)
    for c in main_df.columns:
        # Try convert
        try:
            converted = pd.to_numeric(main_df[c][:20], errors='raise')
            numeric_fields.append(c)
        except (ValueError, TypeError):
            continue
if not numeric_fields:
    raise RuntimeError("No numeric fields found to demonstrate EDA.")
# Use the first numeric field
numeric_field_id = numeric_fields[0]
print(f"Using numeric field (@id): {numeric_field_id}")

# Clean/convert (if needed) and analyze
main_df[numeric_field_id] = pd.to_numeric(main_df[numeric_field_id], errors='coerce')

# Filtering
median_value = main_df[numeric_field_id].median()
filtered_df = main_df[main_df[numeric_field_id] > median_value]
print(f"Filtered records with {numeric_field_id} > median ({median_value}):\n")
print(filtered_df[[numeric_field_id]].head())

# Normalization (Z-score)
filtered_df[f"{numeric_field_id}_normalized"] = (
    filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
) / filtered_df[numeric_field_id].std()
print(f"\nNormalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Group by a categorical field if present
categorical_fields = [c for c in main_df.columns if main_df[c].dtype == 'object' and c != numeric_field_id]
if categorical_fields:
    group_field = categorical_fields[0]
    print(f"\nGrouping by field (@id): {group_field}")
    grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
    print(grouped_df[[numeric_field_id]].head())
else:
    print("No suitable categorical/group field found to group by.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

We'll plot the distribution of the selected numeric field and, if available, a boxplot grouped by the selected group field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8, 4))
sns.histplot(main_df[numeric_field_id].dropna(), kde=True)
plt.title(f"Distribution of field: {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.show()

# Boxplot by group field if available
if categorical_fields:
    plt.figure(figsize=(10, 5))
    group_field = categorical_fields[0]
    sns.boxplot(data=main_df, x=group_field, y=numeric_field_id)
    plt.title(f"Boxplot of {numeric_field_id} by {group_field}")
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion
This notebook demonstrated how to use `mlcroissant` to load a dataset defined by a Croissant schema, explore its record sets and fields by `@id`, and perform basic exploratory data analysis and visualization. You can extend this notebook to perform deeper domain-specific analysis on the FAIR² dataset or apply similar steps to other Croissant datasets.
